# 🩺 Alzheimer diagnosis from Spanish audio

Full pipeline: Whisper transcription → cognitive feature extraction (T2–T7) → BERT / Wav2Vec2 semantic embeddings.

## 🛠️ Setup e imports

In [ ]:
from src.config import AUTH_TOKEN_HF
AUTH_TOKEN_HF


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch

print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Nº GPUs:        {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device:  {device}")

## ⚙️ Path configuration

In [ ]:
from src.utils import DATA_DIR
print(DATA_DIR)


## Extract ZIP

In [ ]:
import os
import subprocess
from src.utils import DATA_DIR, DATA_EXTRACTED_DIR

PATH_7ZIP = "C:\\Program Files\\7-Zip\\7z.exe"

def extract_with_7zip(base_name, output_dir):
    """Extract a multi-part ZIP archive with 7zip"""
    first_volume = os.path.join(DATA_DIR, f"{base_name}.z01")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Extracting {base_name}...")
    result = subprocess.run(
        [PATH_7ZIP, "x", first_volume, f"-o{output_dir}", "-y"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    else:
        print(f"Successfully extracted en {output_dir}")

extract_with_7zip("audio_controls", os.path.join(DATA_EXTRACTED_DIR,'audio', "controls"))
extract_with_7zip("audio_patients", os.path.join(DATA_EXTRACTED_DIR,'audio', "patients"))

In [ ]:
import os
import subprocess
from src.utils import DATA_DIR, DATA_EXTRACTED_DIR

PATH_7ZIP = "C:\\Program Files\\7-Zip\\7z.exe"  # adjust if needed

def extract_with_7zip(zip_name, output_dir):
    path_zip = os.path.join(DATA_DIR, zip_name)
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Extracting {zip_name}...")
    result = subprocess.run(
        [PATH_7ZIP, "x", path_zip, f"-o{output_dir}", "-y"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    else:
        print(f"✅ Successfully extracted en {output_dir}")

extract_with_7zip("labels_controls.zip", os.path.join(DATA_EXTRACTED_DIR,"labels","controls"))
extract_with_7zip("labels_patients.zip", os.path.join(DATA_EXTRACTED_DIR,"labels","patients"))

## 📂 Data loading and exploration

In [ ]:
import os
from src.utils import PATH_DIR_AUDIO_CONTROLS, PATH_DIR_AUDIO_PATIENTS
def listar_wavs(directorio):
    """Return a list of .wav files in a directory."""
    if not os.path.isdir(directorio):
        return []
    return sorted(f for f in os.listdir(directorio) if f.endswith(".wav"))

control_wavs = listar_wavs(PATH_DIR_AUDIO_CONTROLS)
patient_wavs = listar_wavs(PATH_DIR_AUDIO_PATIENTS)

print(f"Controls: {len(control_wavs)} files")
print(f"Patients: {len(patient_wavs)} files")

In [ ]:
import pandas as pd
from src.utils import PATH_METADATA
information_data = pd.read_csv(PATH_METADATA, sep=";")
information_data.head()

##  Audio transcription

In [ ]:
import whisper
def clear_hooks(model):
    """Elimina todos los forward hooks registrados en el modelo (p. ej. por word_timestamps)."""
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()

MODEL_WHISPER_NAME = "large-v3-turbo"
whisper_model = whisper.load_model(MODEL_WHISPER_NAME, device=device)
clear_hooks(whisper_model)   # ← limpia hooks residuales de ejecuciones anteriores
print(f"Whisper '{MODEL_WHISPER_NAME}' loaded on {device}")

In [ ]:
from src.transcriptor import transcribe_audio
from src.utils import PATH_TRANSCRIPTIONS, apply_function_to_dataset

df_transcriptions = apply_function_to_dataset(transcribe_audio, debug=False)
df_transcriptions.to_csv(PATH_TRANSCRIPTIONS, index=False)


## 📝 Information extraction

Transcriptions are loaded and merged with metadata to extract the results of each cognitive task (T2–T7).

In [ ]:
import pandas as pd
from src.utils import PATH_METADATA, PATH_TRANSCRIPTIONS
df_transcriptionses = pd.read_csv(PATH_TRANSCRIPTIONS, index_col=0)

information_data = pd.read_csv(PATH_METADATA, sep=";")

df = df_transcriptionses.join(
    information_data.set_index("Participant ID"),
    on=df_transcriptionses.index
)

columnas_relevantes = [
    "T1", "T2", "T3", "T4", "T5", "T6", "T7",
    "Group", "Diagnosis (only patients)", "Date (mm/dd/yy)",
]
df = df[columnas_relevantes]
print(f"Dataset: {df.shape[0]} observaciones, {df.shape[1]} columnas")
df.head()

## Evaluate patient tasks

In [ ]:
import os
import pandas as pd
from src.utils import DATA_EXTRACTED_DIR, PATH_DIR_LABELS_CONTROLS, PATH_DIR_LABELS_PATIENTS, PATH_RESULTS
from src.evaluate_transcription import EVALUATOR

evaluator = EVALUATOR()

results_by_id = {}
for index,row in df.iterrows():
    print(f"Evaluating patient {index}...")

    if row['Group'] == 'Patient':
        path_label = os.path.join(PATH_DIR_LABELS_PATIENTS, f"{index}.txt")
    else:
        path_label = os.path.join(PATH_DIR_LABELS_CONTROLS, f"{index}.txt")
    
    
    results = evaluator.evaluate_patient(row,path_label, debug=False)
    print(results)
    results_by_id[index] = results

PATH_RESULTS  = os.path.join(DATA_EXTRACTED_DIR, "final_results.csv")
df_results = pd.DataFrame.from_dict(results_by_id, orient="index")
df_results.to_csv(PATH_RESULTS, index=False)

## Extract embeddings

In [ ]:
from src.embedding_features import ExtractEmbeddings
from src.get_dataset_info import PATH_EMBEDDINGS
from src.utils import apply_function_to_dataset

embedding_extractor = ExtractEmbeddings()
df_embeddings = apply_function_to_dataset(embedding_extractor.extract_embeddings)
df_embeddings.to_csv(PATH_EMBEDDINGS, index=False)

## Extract audio and transcription features

In [ ]:
from src.features import extract_features
from src.get_dataset_info import PATH_FEATURES
from src.utils import apply_function_to_dataset

df_features = apply_function_to_dataset(extract_features)
df_features.to_csv(PATH_FEATURES, index=False)